# 03 — Phase 3: stage-2 GRPO (Simulation) from each stage-1 checkpoint

**Committed grid: 3 checkpoints x seed 42 = 3 cells** (plan §3 Phase 3). Group 8, reward `exact` (no shaping bonus for Simulation — plan §1.4). Do not start the stretch-goal seeds (43/44) until these 3 cells are complete, validated, and Phase 4 has produced a readable result.

In [ ]:
import subprocess, sys, os, json
from pathlib import Path

# Private repo: reads a token from Colab's own Secrets store (key icon,
# left sidebar) — add one named GITHUB_TOKEN (a GitHub PAT with repo read
# access) before running this cell. The token is never written to this
# notebook's source and this cell never prints it.
from google.colab import userdata
try:
    _token = userdata.get('GITHUB_TOKEN')
except Exception:
    _token = None
if not _token:
    raise RuntimeError(
        'Add a GITHUB_TOKEN secret (key icon, left sidebar) with repo read '
        'access to WYR186/RLVR, enable notebook access for it, then re-run.')
REPO_URL = f'https://{_token}@github.com/WYR186/RLVR.git'
REPO_DIR = '/content/RLVR'

def _run_git(args):
    # NEVER let a failed git command's exception propagate raw: a
    # CalledProcessError's default message includes the full argv,
    # which for the clone/pull commands contains the embedded token —
    # that leaked into a cell's visible output the first time this ran
    # without this wrapper. Capture output and re-raise sanitized.
    result = subprocess.run(args, capture_output=True, text=True)
    if result.returncode != 0:
        safe_args = [a.replace(_token, '***REDACTED***') for a in args]
        safe_stderr = (result.stderr or '').replace(_token, '***REDACTED***')
        raise RuntimeError(f'{safe_args} failed (exit {result.returncode}): {safe_stderr}') from None
    return result

if not os.path.isdir(REPO_DIR):
    _run_git(['git', 'clone', REPO_URL, REPO_DIR])
else:
    _run_git(['git', '-C', REPO_DIR, 'pull'])
# strip the token back out of the stored remote URL immediately — no need
# to leave it sitting in .git/config for the rest of the session
_run_git(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin',
         'https://github.com/WYR186/RLVR.git'])
del _token, REPO_URL  # don't leave the token bound in the notebook's live namespace

EXP2_DIR = f'{REPO_DIR}/experiment 2'
sys.path.insert(0, EXP2_DIR)  # only this one goes on sys.path — pipeline.py
# reaches eaaj-pilot/src by explicit file path internally, avoiding a
# top-level `src` package-name collision between the two sibling dirs
# (see experiment 2/src/pipeline.py's module docstring).

import src.guru_data as guru_data
import src.guru_reward as guru_reward
import src.pipeline as pipeline

CONFIG = json.load(open(f'{EXP2_DIR}/exp2_colab_config.json'))
DATA_DIR = Path(EXP2_DIR) / 'data'
MODEL_ID, MODEL_REVISION = CONFIG['model_id'], CONFIG['model_revision']
DATASET_REVISION = CONFIG['dataset']['revision']
print('config loaded:', CONFIG['experiment'])
print('merge note:', CONFIG['merge_note'])

In [ ]:
%pip install -q -r "/content/RLVR/experiment 2/requirements.txt"

In [ ]:
import gc, torch
splits = json.load(open(DATA_DIR / 'exp2_colab_splits.json'))
stage_b_train_rows = guru_data.dataset_rows_for(
    'b', 'train', splits, MODEL_ID, MODEL_REVISION, DATASET_REVISION)
stage_b_train = guru_data.to_hf_dataset(stage_b_train_rows)
eval_rows = guru_data.dataset_rows_for(
    'b', 'eval', splits, MODEL_ID, MODEL_REVISION, DATASET_REVISION)

RUN_DIR = f'{EXP2_DIR}/../eaaj-pilot/outputs/exp2_colab_guru_math7b_group8_REPLACE_WITH_HASH'
STAGE_A_DIR = f'{RUN_DIR}/stage_a'
sb = CONFIG['stage_b']
sa = CONFIG['stage_a']
EVAL_EVERY = sb['eval_at_updates'][1] - sb['eval_at_updates'][0]
assert EVAL_EVERY > 0

## Committed grid: 3 checkpoints x seed 42

In [ ]:
results = {}
for step in sa['adapt_from_checkpoints']:
    for seed in sb['committed_seeds']:
        cell_dir = f'{RUN_DIR}/stage_b/ckpt{step}_seed{seed}'
        print(f'=== ckpt {step}, seed {seed} ===')
        summary = pipeline.run_stage_b_adaptation(
            MODEL_ID, CONFIG['peft'], f'{STAGE_A_DIR}/ckpt-{step}',
            stage_b_train, eval_rows, cell_dir,
            budget_updates=sb['budget_updates'], eval_every=EVAL_EVERY,
            reward_mode=sb['reward_mode'],
            learning_rate=sb['learning_rate'], per_device_batch=sb['per_device_train_batch_size'],
            grad_accum=sb['gradient_accumulation_steps'], num_generations=sb['num_generations'],
            beta=sb['beta'], temperature=sb['temperature'], top_p=sb['top_p'],
            max_completion_length=sb['max_completion_length'],
            revision=MODEL_REVISION, seed=seed)
        results[f'ckpt{step}_seed{seed}'] = summary
        print(summary)
        gc.collect(); torch.cuda.empty_cache()

print('Committed grid complete:', list(results.keys()))

## Stretch goal (only after the committed grid is complete and reported)

Uncomment and run only if Phase 0's measured throughput leaves runway before 2026-08-23 (plan §4 compute budget).

In [ ]:
# for step in sa['adapt_from_checkpoints']:
#     for seed in sb['stretch_goal_seeds']:
#         cell_dir = f'{RUN_DIR}/stage_b/ckpt{step}_seed{seed}'
#         summary = pipeline.run_stage_b_adaptation(
#             MODEL_ID, CONFIG['peft'], f'{STAGE_A_DIR}/ckpt-{step}',
#             stage_b_train, eval_rows, cell_dir,
#             budget_updates=sb['budget_updates'], eval_every=EVAL_EVERY,
#             reward_mode=sb['reward_mode'],
#             learning_rate=sb['learning_rate'], per_device_batch=sb['per_device_train_batch_size'],
#             grad_accum=sb['gradient_accumulation_steps'], num_generations=sb['num_generations'],
#             beta=sb['beta'], temperature=sb['temperature'], top_p=sb['top_p'],
#             max_completion_length=sb['max_completion_length'],
#             revision=MODEL_REVISION, seed=seed)
#         results[f'ckpt{step}_seed{seed}'] = summary
#         gc.collect(); torch.cuda.empty_cache()

## Commit reminder

Commit `stage_b/` (every cell's dashboard/curve/summary), prefix `exp2-colab:`, one commit per pass (committed grid, then stretch goal if run). Never overwrite a preserved `*_oom_*` directory.